# `16 — Hashing theory + universal families + polynomial hash`

## **Hash definition**
A hash is a function:
$h: K \to V$, typically $V = \{0, 1, \ldots, M-1\}$ and $M \ll |K|$.

Because $V$ is smaller than $K$, collisions must exist:
- distinct keys $x \neq y$ may have $h(x) = h(y)$ (collision).

Define collision indicator:
$$C_h(x, y) = \begin{cases} 1 & \text{if } x \neq y \text{ and } h(x)=h(y) \\ 0 & \text{otherwise} \end{cases}$$

---

## **Proof: expected collisions is N/M for a random hash**
Let:
- fixed query key $x$
- set $S$ of $N$ keys (none equal to $x$)
- choose hash function $h$ uniformly at random from all functions $K \to V$

We want:
$$\mathbb{E}\left[ \sum_{s \in S} C_h(s, x) \right]$$

Linearity of expectation:
$$\mathbb{E}\left[ \sum C \right] = \sum \mathbb{E}[C] = \sum P(C=1)$$

For a truly random function:
$$P(h(s) = h(x)) = \frac{1}{M} \quad \text{(uniform over } V\text{)}$$

So:
$$\mathbb{E}\left[ \sum_{s \in S} C_h(s, x) \right] = \sum_{s \in S} \frac{1}{M} = \frac{N}{M}$$

If $M \geq N$ then expected collisions $\leq 1$, which is great.

---

## **Universal families**
We cannot store a fully random function (too many).
Instead we use a family $\mathcal{H}$ of functions and pick $h$ uniformly from $\mathcal{H}$.

$\mathcal{H}$ is **universal** if for any $x \neq y$:
$$P_{h \in \mathcal{H}}[h(x)=h(y)] \leq \frac{1}{M}$$

Example for integer keys:
Pick a prime $p \geq |K|$
$$h_{a,b}(x) = ((a \cdot x + b) \bmod p) \bmod M$$
where $a \neq 0$, $a, b \in [0, p)$

This family is universal.

---

## **Polynomial hash for strings**
For string $s = s_0 s_1 \ldots s_{L-1}$ (characters mapped to integers),
choose:
- large prime $p$
- base $a$ (random in $[1, p)$)

$$\text{hash}(s) = \sum_{i=0}^{L-1} s_i \cdot a^i \pmod{p}$$

This supports fast substring hashes using prefix hashes (like prefix sums),
and is the foundation for Rabin–Karp.


In [2]:
#  Visual simulation: collisions ≈ N/M (Monte Carlo)
import random
from typing import List, Tuple


def count_collisions_for_x(keys: List[int], *, x: int, M: int, h) -> int:
    hx = h(x)
    return sum(1 for k in keys if k != x and h(k) == hx)


def simulate_expected_collisions(*, N: int, M: int, trials: int = 2000, seed: int = 7) -> None:
    """
    Keys are integers. Hash is chosen as a random function by random table values
    over a finite domain for the demo.
    """
    random.seed(seed)

    # Finite domain just for simulation
    domain_size = max(10 * N, 200)
    domain = list(range(domain_size))

    # Pick x and a set S of N keys not equal to x
    x = random.choice(domain)
    keys = [k for k in random.sample(domain, N + 1) if k != x][:N]

    collisions: List[int] = []

    for _ in range(trials):
        # Simulate a "random function" on this finite domain via random mapping
        table = [random.randrange(0, M) for _ in range(domain_size)]
        h = lambda t, table=table: table[t]
        c = count_collisions_for_x(keys, x=x, M=M, h=h)
        collisions.append(c)

    avg = sum(collisions) / len(collisions)

    print("-" * 70)
    print(f"Simulating E[#collisions] for random h:K->V with |V|=M")
    print("-" * 70)
    print(f"N={N}, M={M}, trials={trials}")
    print(f"theoretical N/M = {N/M:.4f}")
    print(f"empirical average = {avg:.4f}")
    print(f"min={min(collisions)}, max={max(collisions)}")


simulate_expected_collisions(N=200, M=400, trials=1000)
simulate_expected_collisions(N=200, M=200, trials=1000)
simulate_expected_collisions(N=200, M=100, trials=1000)


----------------------------------------------------------------------
Simulating E[#collisions] for random h:K->V with |V|=M
----------------------------------------------------------------------
N=200, M=400, trials=1000
theoretical N/M = 0.5000
empirical average = 0.5210
min=0, max=4
----------------------------------------------------------------------
Simulating E[#collisions] for random h:K->V with |V|=M
----------------------------------------------------------------------
N=200, M=200, trials=1000
theoretical N/M = 1.0000
empirical average = 1.0090
min=0, max=6
----------------------------------------------------------------------
Simulating E[#collisions] for random h:K->V with |V|=M
----------------------------------------------------------------------
N=200, M=100, trials=1000
theoretical N/M = 2.0000
empirical average = 2.0030
min=0, max=8


In [3]:
# Universal hashing for integers: h_{a,b}(x) = ((a*x + b) mod p) mod M
import random
from typing import Callable, List


def make_universal_hash(*, p: int, M: int, a: int, b: int) -> Callable[[int], int]:
    def h(x: int) -> int:
        return ((a * x + b) % p) % M
    return h


def demo_universal_family(*, keys: List[int], x: int, p: int, M: int, trials: int = 30, seed: int = 7) -> None:
    random.seed(seed)
    print("-" * 70)
    print("Universal family demo: random (a,b) => collision count changes")
    print("-" * 70)
    print(f"x={x}, N={len(keys)}, p={p}, M={M}")
    print(f"Expected collisions about N/M = {len(keys)/M:.3f}")
    print("-" * 70)

    for t in range(trials):
        a = random.randrange(1, p)   # a != 0
        b = random.randrange(0, p)
        h = make_universal_hash(p=p, M=M, a=a, b=b)
        c = count_collisions_for_x(keys, x=x, M=M, h=h)
        print(f"trial={t:>2}: a={a:>3}, b={b:>3} -> collisions={c}")


keys = list(range(50, 250))  # N=200
x = 17
demo_universal_family(keys=keys, x=x, p=257, M=211, trials=12)


----------------------------------------------------------------------
Universal family demo: random (a,b) => collision count changes
----------------------------------------------------------------------
x=17, N=200, p=257, M=211
Expected collisions about N/M = 0.948
----------------------------------------------------------------------
trial= 0: a=166, b= 77 -> collisions=0
trial= 1: a=203, b= 24 -> collisions=0
trial= 2: a= 38, b= 48 -> collisions=0
trial= 3: a=188, b= 29 -> collisions=0
trial= 4: a=110, b= 19 -> collisions=0
trial= 5: a= 45, b=222 -> collisions=1
trial= 6: a=215, b= 35 -> collisions=0
trial= 7: a=124, b= 46 -> collisions=0
trial= 8: a=218, b= 30 -> collisions=0
trial= 9: a= 64, b=114 -> collisions=0
trial=10: a= 32, b=203 -> collisions=1
trial=11: a= 26, b=113 -> collisions=0


In [4]:
# Polynomial hash for strings + prefix-hash visualization
from typing import List, Tuple


def char_to_int(c: str) -> int:
    # Simple mapping (works for lowercase a..z)
    return ord(c) - ord("a") + 1


def build_poly_prefix_hash(s: str, *, a: int, p: int, verbose: bool = True) -> Tuple[List[int], List[int]]:
    """
    h[i] = hash(s[:i]) = sum_{j=0..i-1} s[j]*a^j mod p
    pow_a[i] = a^i mod p
    """
    n: int = len(s)
    h: List[int] = [0] * (n + 1)
    pow_a: List[int] = [1] * (n + 1)

    if verbose:
        print("-" * 80)
        print("Building polynomial prefix hash")
        print(f"s={s!r}, base a={a}, mod p={p}")
        print("-" * 80)

    for i in range(n):
        h[i + 1] = (h[i] + char_to_int(s[i]) * pow_a[i]) % p
        pow_a[i + 1] = (pow_a[i] * a) % p
        if verbose:
            print(f"i={i:>2} char={s[i]!r} val={char_to_int(s[i])} "
                  f"-> h[{i+1}]={h[i+1]}, a^{i+1}={pow_a[i+1]}")

    return h, pow_a


def substring_hash(h: List[int], pow_a: List[int], *, l: int, r: int, p: int) -> int:
    """
    Returns hash(s[l:r]) normalized as sum s[i]*a^(i-l) mod p.
    We use:
      h[r] - h[l] = sum_{i=l..r-1} s[i]*a^i
    Divide by a^l => multiply by inv(a^l) in field.
    To avoid inverses, in Rabin–Karp we compare multiplied values instead.
    Here, for teaching, we DO normalize using pow_a and modular inverse.
    """
    # Fermat inverse (p prime): inv(x) = x^(p-2) mod p
    raw = (h[r] - h[l]) % p
    inv = pow(pow_a[l], p - 2, p)
    return (raw * inv) % p


s = "ababacababa"
a = 911382323
p = 1_000_000_007

h, pow_a = build_poly_prefix_hash(s, a=a, p=p, verbose=True)

print("-" * 80)
print("Substring hash demo:")
print("s[2:5] =", s[2:5])
print("hash(s[2:5]) =", substring_hash(h, pow_a, l=2, r=5, p=p))
print("s[0:3] =", s[0:3])
print("hash(s[0:3]) =", substring_hash(h, pow_a, l=0, r=3, p=p))


--------------------------------------------------------------------------------
Building polynomial prefix hash
s='ababacababa', base a=911382323, mod p=1000000007
--------------------------------------------------------------------------------
i= 0 char='a' val=1 -> h[1]=1, a^1=911382323
i= 1 char='b' val=2 -> h[2]=822764640, a^2=862552205
i= 2 char='a' val=1 -> h[3]=685316838, a^3=798868433
i= 3 char='b' val=2 -> h[4]=283053690, a^4=142387918
i= 4 char='a' val=1 -> h[5]=425441608, a^5=565584704
i= 5 char='c' val=3 -> h[6]=122195706, a^6=776540113
i= 6 char='a' val=1 -> h[7]=898735819, a^7=134548010
i= 7 char='b' val=2 -> h[8]=167831832, a^8=50454491
i= 8 char='a' val=1 -> h[9]=218286323, a^9=891479283
i= 9 char='b' val=2 -> h[10]=1244875, a^10=159565231
i=10 char='a' val=1 -> h[11]=160810106, a^11=880837110
--------------------------------------------------------------------------------
Substring hash demo:
s[2:5] = aba
hash(s[2:5]) = 685316838
s[0:3] = aba
hash(s[0:3]) = 685316838


## **Final summary:**

### **Hashing**
- Expected collisions for random function: **E = N/M**
- Universal family guarantees: **P(collision) ≤ 1/M**
- Polynomial hash supports fast substring hashing.

### **Rabin–Karp**
- Rolling hash compares substring hashes in O(1)
- Total expected: **O(|s| + |t| + verification work)**

### **Hash tables**
| Method | Collision handling | Expected op time (α=N/M) | Notes |
|---|---|---:|---|
| Separate chaining | linked list per bucket | **O(α+1)** | easy deletes |
| Open addressing | probing in array | depends strongly on α | keep α small; Python uses open addressing + double hashing |